In [ ]:
# step1_search_api_adaptive.py

import os
import requests
import pandas as pd
from dotenv import load_dotenv
from datetime import datetime, timedelta
from time import sleep

# === Load tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 6)]
tokens = [t for t in tokens if t]
if not tokens:
    raise ValueError("No tokens found in All_Token.env")

token_index = 0
def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {
        "Authorization": f"token {token}",
        "Accept": "application/vnd.github.v3+json",
        "User-Agent": "android-repo-crawler/1.0"
    }

def run_query(created_range, lang, page):
    url = (
        f"https://api.github.com/search/repositories"
        f"?q=stars:>0+fork:false+archived:false+language:{lang}+created:{created_range}"
        f"&per_page=100&page={page}"
    )
    response = requests.get(url, headers=get_headers())
    if response.status_code != 200:
        print(f"Error {response.status_code}: {response.text}")
        sleep(10)
        return None
    return response.json()

# === Adaptive time window loop ===
results = []
start_date = datetime(2008, 1, 1)
end_date = datetime(2024, 12, 31)
initial_window_days = 7
min_window_days = 0.5
max_window_days = 30

languages = ["Kotlin", "Java", "Dart"]

for lang in languages:
    current = start_date
    window_days = initial_window_days

    while current < end_date:
        next_date = current + timedelta(days=window_days)
        if next_date > end_date:
            next_date = end_date

        created_range = f"{current.date()}..{next_date.date()}"
        total_fetched = 0
        page = 1
        items = []

        while page <= 10:  # GitHub Search API only allows up to 1000 results (10 pages)
            data = run_query(created_range, lang, page)
            if data is None:
                break
            fetched = data.get("items", [])
            if not fetched:
                break
            items.extend(fetched)
            total_fetched += len(fetched)
            print(f"{lang} | {created_range} | Page {page} | Fetched {len(fetched)}")
            if len(fetched) < 100:
                break
            page += 1
            sleep(1)

        # Store data
        for repo in items:
            results.append({
                "full_name": repo["full_name"],
                "html_url": repo["html_url"],
                "language": repo["language"],
                "created_at": repo["created_at"],
                "description": repo.get("description", ""),
                "topics": ",".join(repo.get("topics", [])),
                "name": repo.get("name", ""),
                "stars": repo.get("stargazers_count", 0),
            })

        # Adapt window size based on count
        if total_fetched >= 1000 and window_days > min_window_days:
            window_days = max(min_window_days, window_days / 2)
            print(f"⬇️ Shrinking window to {window_days} days")
        elif total_fetched < 300 and window_days < max_window_days:
            window_days = min(max_window_days, window_days * 2)
            print(f"⬆️ Expanding window to {window_days} days")
        else:
            current = next_date

# === Save Output ===
output_dir = r"C:\Android Mobile App\Step1_URL_Search\Type_2_Searching_Pipeline"
os.makedirs(output_dir, exist_ok=True)
output_path = os.path.join(output_dir, "step1_search_output.csv")

df = pd.DataFrame(results)
df.to_csv(output_path, index=False)
print(f"✅ Step 1 complete. Saved to: {output_path}")


ValueError: No tokens found in All_Token.env